# Fetch SEER StatFacts — Update Curated YAMLs

Fetches annual epidemiology data from SEER Cancer StatFacts HTML pages and writes  
gold-tier values back into the curated YAML files (`config/curated_data/*.yaml`).

**Updates per YAML:**
- `incidence_rate_{year}` — SEER 12 preferred, SEER 8 fallback
- `mortality_rate_{year}` — U.S. Observed
- `five_year_survival_period_{year}` — SEER 8 Observed
- `incidence_rate_{race}` — 2020–2024 cross-sectional
- `mortality_rate_{race}` — 2020–2024 cross-sectional
- `stage_localized_pct`, `stage_regional_pct`, `stage_distant_pct`

All other existing YAML keys are preserved unchanged.

**Usage:** Set `TARGET_INDICATIONS` in Cell 2, then Run All.

In [ ]:
# Cell 1 — Imports
import sys
import re
import requests
import yaml
from pathlib import Path
from bs4 import BeautifulSoup

print('Imports OK')

In [ ]:
# Cell 2 — Configuration
# ── Edit to run for specific indications or leave empty for all ──
TARGET_INDICATIONS = []  # e.g. ["cll", "prostate"] or [] for all 6

ROOT = Path().resolve().parent  # project root (one level up from scripts/)

SEER_PAGES = {
    "cll": {
        "url":        "https://seer.cancer.gov/statfacts/html/clyl.html",
        "yaml":       ROOT / "config/curated_data/cll.yaml",
        "label":      "Chronic Lymphocytic Leukemia",
        "population": "All ages, both sexes",
    },
    "hodgkin": {
        "url":        "https://seer.cancer.gov/statfacts/html/hodg.html",
        "yaml":       ROOT / "config/curated_data/hodgkin.yaml",
        "label":      "Hodgkin Lymphoma",
        "population": "All ages, both sexes",
    },
    "nhl": {
        "url":        "https://seer.cancer.gov/statfacts/html/nhl.html",
        "yaml":       ROOT / "config/curated_data/nhl.yaml",
        "label":      "Non-Hodgkin Lymphoma",
        "population": "All ages, both sexes",
    },
    "gc": {
        "url":        "https://seer.cancer.gov/statfacts/html/stomach.html",
        "yaml":       ROOT / "config/curated_data/gc.yaml",
        "label":      "Stomach (Gastric) Cancer",
        "population": "All ages, both sexes",
    },
    "ovarian": {
        "url":        "https://seer.cancer.gov/statfacts/html/ovary.html",
        "yaml":       ROOT / "config/curated_data/ovarian.yaml",
        "label":      "Ovarian Cancer",
        "population": "All ages, females",
    },
    "prostate": {
        "url":        "https://seer.cancer.gov/statfacts/html/prost.html",
        "yaml":       ROOT / "config/curated_data/prostate.yaml",
        "label":      "Prostate Cancer",
        "population": "All ages, males",
    },
}

RACE_KEY = {
    "All Races":                                 "all_races",
    "Hispanic":                                  "hispanic",
    "Non-Hispanic American Indian/Alaska Native": "aian",
    "Non-Hispanic Asian/Pacific Islander":       "asian_pacific_islander",
    "Non-Hispanic Black":                        "black",
    "Non-Hispanic White":                        "white_non_hispanic",
}

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-data-pipeline/1.0)"}

targets = TARGET_INDICATIONS or list(SEER_PAGES.keys())
print(f'Target indications: {targets}')
print(f'Project root: {ROOT}')

In [ ]:
# Cell 3 — Helper functions

def _clean(val: str) -> str:
    return val.strip().rstrip("%").strip()

def _valid(val: str) -> bool:
    v = _clean(val)
    return bool(v) and v != "-"

def _entry(value, unit, year, population, definition, citation, url,
           category="Incidence Rate", geography="US"):
    return {
        "value":           str(value),
        "unit":            unit,
        "year":            str(year),
        "geography":       geography,
        "population":      population,
        "definition":      definition,
        "source_citation": citation,
        "source_url":      url,
        "tier":            "gold",
        "confidence":      "high",
        "category":        category,
    }

def fetch_page(url: str) -> BeautifulSoup:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

print('Helpers defined')

In [ ]:
# Cell 4 — Table parsers

def parse_annual_trends(soup, cfg):
    """scrapeTable_01: year × (inc rate, mort rate, 5-yr survival)."""
    table = soup.find("table", {"id": "scrapeTable_01"})
    if not table:
        print(f"  WARNING: scrapeTable_01 not found")
        return {}
    url, label, pop = cfg["url"], cfg["label"], cfg["population"]
    citation = f"SEER Cancer Stat Facts: {label}. National Cancer Institute. Bethesda, MD, 2024"
    entries = {}
    for row in table.find_all("tr"):
        cells = [td.get_text(strip=True) for td in row.find_all("td")]
        if len(cells) < 6 or not cells[0].isdigit():
            continue
        year    = cells[0]
        inc_obs = cells[3] if _valid(cells[3]) else cells[1]
        mort    = cells[5] if len(cells) > 5 else ""
        surv    = cells[7] if len(cells) > 7 and _valid(cells[7]) else (cells[8] if len(cells) > 8 else "")
        reg     = "SEER 12" if _valid(cells[3]) else "SEER 8"
        if _valid(inc_obs):
            entries[f"incidence_rate_{year}"] = _entry(_clean(inc_obs), f"per 100k/yr ({reg})",
                year, pop, f"Age-adjusted incidence rate — {reg}", citation, url, "Incidence Rate")
        if _valid(mort):
            entries[f"mortality_rate_{year}"] = _entry(_clean(mort), "per 100k/yr (U.S.)",
                year, pop, "Age-adjusted mortality rate — United States", citation, url, "Mortality Rate")
        if _valid(surv):
            entries[f"five_year_survival_period_{year}"] = _entry(_clean(surv), "% 5-year relative survival",
                year, pop, "5-year relative survival — SEER 8", citation, url, "Survival")
    return entries


def parse_race_incidence(soup, cfg):
    """scrapeTable_04: race × incidence rate (2020-2024)."""
    table = soup.find("table", {"id": "scrapeTable_04"})
    if not table:
        return {}
    url, label, pop = cfg["url"], cfg["label"], cfg["population"]
    citation = f"SEER Cancer Stat Facts: {label}. NCI, 2024 (2020–2024 average)"
    entries = {}
    for row in table.find_all("tr"):
        cells = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        if len(cells) < 2 or not _valid(cells[1]) or "sex-specific" in cells[1].lower():
            continue
        race_key = RACE_KEY.get(cells[0].strip())
        if race_key:
            entries[f"incidence_rate_{race_key}"] = _entry(cells[1], "per 100k/yr",
                "2020-2024", f"{pop}, {cells[0].strip()}",
                f"Age-adjusted incidence rate — {cells[0].strip()} — 2020-2024", citation, url, "Incidence Rate")
    return entries


def parse_race_mortality(soup, cfg):
    """scrapeTable_07: race × mortality rate (2020-2024)."""
    table = soup.find("table", {"id": "scrapeTable_07"})
    if not table:
        return {}
    url, label, pop = cfg["url"], cfg["label"], cfg["population"]
    citation = f"SEER Cancer Stat Facts: {label}. NCI, 2024 (2020–2024 average)"
    entries = {}
    for row in table.find_all("tr"):
        cells = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        if len(cells) < 2 or not _valid(cells[1]) or "sex-specific" in cells[1].lower():
            continue
        race_key = RACE_KEY.get(cells[0].strip())
        if race_key:
            entries[f"mortality_rate_{race_key}"] = _entry(cells[1], "per 100k/yr",
                "2020-2024", f"{pop}, {cells[0].strip()}",
                f"Age-adjusted mortality rate — {cells[0].strip()} — 2020-2024", citation, url, "Mortality Rate")
    return entries


def parse_stage_distribution(soup, cfg):
    """scrapeTable_02: stage % and 5-yr survival."""
    table = soup.find("table", {"id": "scrapeTable_02"})
    if not table:
        return {}
    url, label = cfg["url"], cfg["label"]
    citation = f"SEER Cancer Stat Facts: {label}. NCI, 2024"
    STAGE_MAP = {"localized": "stage_localized", "regional": "stage_regional",
                 "distant": "stage_distant", "unknown": "stage_unknown"}
    entries = {}
    for row in table.find_all("tr"):
        cells = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        if len(cells) < 2:
            continue
        stage_raw = cells[0].lower()
        pct_raw   = _clean(cells[1]) if len(cells) > 1 else ""
        surv_raw  = _clean(cells[2]) if len(cells) > 2 else ""
        for frag, prefix in STAGE_MAP.items():
            if stage_raw.startswith(frag):
                if _valid(pct_raw):
                    entries[f"{prefix}_pct"] = _entry(pct_raw, "% of diagnosed", "2024",
                        cfg["population"], f"Proportion at {frag} stage", citation, url, "Stage Distribution")
                if _valid(surv_raw):
                    entries[f"{prefix}_5yr_survival"] = _entry(surv_raw, "% 5-yr survival",
                        "2024", cfg["population"], f"5-yr survival at {frag} stage", citation, url, "Survival")
                break
    return entries


print('Parsers defined')

In [ ]:
# Cell 5 — Run fetcher and update YAMLs

for key in targets:
    cfg = SEER_PAGES[key]
    print(f"\n{'='*60}")
    print(f"  {key.upper()} — {cfg['label']}")
    print(f"  URL: {cfg['url']}")
    print(f"{'='*60}")

    yaml_path = cfg["yaml"]
    if not yaml_path.exists():
        print(f"  ERROR: YAML not found at {yaml_path}")
        continue

    print("  Fetching SEER page...", end=" ", flush=True)
    soup = fetch_page(cfg["url"])
    print("OK")

    new_entries = {}
    annual    = parse_annual_trends(soup, cfg)
    race_inc  = parse_race_incidence(soup, cfg)
    race_mort = parse_race_mortality(soup, cfg)
    stage     = parse_stage_distribution(soup, cfg)
    new_entries.update(annual)
    new_entries.update(race_inc)
    new_entries.update(race_mort)
    new_entries.update(stage)
    print(f"  Parsed: {len(annual)} annual, {len(race_inc)} race-inc, {len(race_mort)} race-mort, {len(stage)} stage")

    with open(yaml_path) as f:
        data = yaml.safe_load(f)
    existing = data.get("metrics", {})
    added = sum(1 for k in new_entries if k not in existing)
    existing.update(new_entries)
    data["metrics"] = existing

    with open(yaml_path, "w") as f:
        yaml.dump(data, f, allow_unicode=True, sort_keys=False, default_flow_style=False, width=120)

    print(f"  Updated {yaml_path.name}: {added} new entries, {len(new_entries)-added} overwritten")
    print(f"  Total metrics: {len(existing)}")

print(f"\n{'='*60}")
print("  Done. All curated YAMLs updated.")
print(f"{'='*60}")